# Run the whole pipeline
One notebook that runs the full chain, in order:
1. `01_raw_to_bronze` (GitHub to Bronze)
2. `02_bronze_to_silver` (Bronze to Silver)
3. `04_silver_to_gold` (Silver to Gold)

Because Gold runs right after Silver, **Gold refreshes by itself every time new data reaches Silver.**

### Three ways to use it
- **Incremental** (normal day): `load_type = incremental`, leave the dates empty. Loads the last 24 hours.
- **Full / history load**: `load_type = full`, leave `since_date` empty (= 12 months ago). The 12 months are cut into windows of `chunk_days` days, so no single load is too big.
- **Backfill**: `load_type = backfill`, type `since_date` and `until_date`.

Every run is logged. The whole chain also writes one row of its own
(`layer = End-to-End`).

In [0]:
%run ./00_setup_and_logging

# Setup and logging
This notebook does two jobs:
1. Makes the schema (database) and all tables, if they don't exist yet.
2. Gives small helper functions that write one row to the log table every time a step runs.

Run this notebook by itself once. Other notebooks call it with `%run`.

Using workspace.dav_project


## Bronze tables
One table per entity (commits, issues, pulls), plus one quarantine table for
records that could not even be given a key. Every table has `load_timestamp`.

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: pipeline_execution_logs


## Phase 3 upgrade of the log table
The log table already exists from Phase 2, so `CREATE TABLE IF NOT EXISTS`
above does not change it. This cell adds the two new columns to an old
table, only if they are missing. Safe to run any number of times.
- `rows_aggregated`: how many source rows were read and rolled up (Silver to Gold).
- `duration_seconds`: how long the step took.

column already there: rows_aggregated
column already there: duration_seconds


## Small helper functions
`new_run_id()` makes one ID for one notebook run.
`log_run(...)` writes one row to the log table. Call it even when a step fails.
`get_merge_metrics(...)` reads how many rows a MERGE just inserted and updated.

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("repos",
    "pandas-dev/pandas,numpy/numpy,scikit-learn/scikit-learn,matplotlib/matplotlib,"
    "scipy/scipy,fastapi/fastapi,pallets/flask,pytest-dev/pytest",
    "Repos (comma separated)")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full", "backfill"], "Load type")
dbutils.widgets.text("since_date", "", "Since date (YYYY-MM-DDTHH:MM:SSZ or YYYY-MM-DD, blank = default)")
dbutils.widgets.text("until_date", "", "Until date (blank = now)")
dbutils.widgets.text("chunk_days", "30", "Window size in days for full/backfill (0 = one single window)")
dbutils.widgets.text("batch_prefix", "", "Batch name prefix (blank = auto)")
dbutils.widgets.dropdown("run_gold", "yes", ["yes", "no"], "Refresh Gold after Silver?")
dbutils.widgets.text("github_token", "", "GitHub token (paste for this run only, CLEAR before saving or exporting)")

REPOS_TEXT = dbutils.widgets.get("repos").strip()
LOAD_TYPE = dbutils.widgets.get("load_type")
SINCE_TEXT = dbutils.widgets.get("since_date").strip()
UNTIL_TEXT = dbutils.widgets.get("until_date").strip()
CHUNK_DAYS = int(dbutils.widgets.get("chunk_days").strip() or "0")
BATCH_PREFIX = dbutils.widgets.get("batch_prefix").strip()
RUN_GOLD = dbutils.widgets.get("run_gold") == "yes"
GITHUB_TOKEN = dbutils.widgets.get("github_token").strip()

## Work out the time windows
A window is one `(since, until)` pair. One window = one run of notebook `01`.

In [0]:
import json
from datetime import datetime, timedelta, timezone

ISO = "%Y-%m-%dT%H:%M:%SZ"


def parse_dt(text):
    """Accepts 2026-09-01T00:00:00Z or 2026-09-01. Returns an aware UTC datetime, or None."""
    if not text:
        return None
    for fmt in (ISO, "%Y-%m-%d"):
        try:
            return datetime.strptime(text, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    raise ValueError(f"Cannot read date '{text}'. Use YYYY-MM-DDTHH:MM:SSZ or YYYY-MM-DD.")


def make_windows(start, end, chunk_days, now):
    """Cuts [start, end] into pieces of chunk_days. end=None means 'up to now' (open end)."""
    if chunk_days <= 0:
        return [(start, end)]
    stop = end or now
    windows, cur = [], start
    while cur < stop:
        nxt = min(cur + timedelta(days=chunk_days), stop)
        is_last = nxt >= stop
        windows.append((cur, None if (is_last and end is None) else nxt))
        cur = nxt
    return windows


PIPELINE_START = now_utc()
SINCE = parse_dt(SINCE_TEXT)
UNTIL = parse_dt(UNTIL_TEXT)

if LOAD_TYPE == "incremental":
    SINCE = SINCE or (PIPELINE_START - timedelta(days=1))
    WINDOWS = [(SINCE, UNTIL)]
elif LOAD_TYPE == "full":
    SINCE = SINCE or (PIPELINE_START - timedelta(days=365))
    WINDOWS = make_windows(SINCE, UNTIL, CHUNK_DAYS, PIPELINE_START)
else:  # backfill
    if SINCE is None or UNTIL is None:
        raise ValueError("backfill needs both since_date and until_date")
    WINDOWS = make_windows(SINCE, UNTIL, CHUNK_DAYS, PIPELINE_START)

PIPE_RUN_ID = new_run_id()
PREFIX = BATCH_PREFIX or f"{LOAD_TYPE}_{PIPELINE_START.strftime('%Y%m%dT%H%M%S')}"
fmt = lambda d: d.strftime(ISO) if d else ""

print(f"pipeline run id: {PIPE_RUN_ID}")
print(f"load_type: {LOAD_TYPE} | windows: {len(WINDOWS)} | run Gold after: {RUN_GOLD}")
for i, (a, b) in enumerate(WINDOWS, 1):
    print(f"  window {i}: {fmt(a)}  ->  {fmt(b) or '(now)'}")

pipeline run id: 4cc99322-52f6-4d09-9b83-ceefb3d02ba1
load_type: incremental | windows: 1 | run Gold after: True
  window 1: 2026-10-23T15:08:08Z  ->  (now)


## Step 1: Raw to Bronze, one run per window
If one window fails, the others still run. Failed windows are listed at the
end, so you know exactly which ones to run again.

In [0]:
def run_child(path, arguments, timeout=7200):
    """Runs another notebook. Returns (ok, parsed_exit_value_or_error_text)."""
    try:
        out = dbutils.notebook.run(path, timeout, arguments)
        try:
            return True, json.loads(out)
        except Exception:
            return True, {"raw": out}
    except Exception as e:
        return False, str(e)


bronze_results, failed_windows = [], []
for i, (a, b) in enumerate(WINDOWS, 1):
    batch = f"{PREFIX}_w{i:02d}"
    print(f"\n--- window {i}/{len(WINDOWS)}: {fmt(a)} -> {fmt(b) or '(now)'}  (batch {batch}) ---")
    ok, out = run_child("./01_raw_to_bronze", {
        "repos": REPOS_TEXT,
        "load_type": "backfill" if LOAD_TYPE != "incremental" else "incremental",
        "since_date": fmt(a),
        "until_date": fmt(b),
        "batch_id": batch,
        "github_token": GITHUB_TOKEN,
    })
    if ok and isinstance(out, dict) and "all_ok" not in out:
        # 01 finished without raising, but did not send its result message back.
        # Do not guess: read what 01 itself wrote in the log table for this batch.
        rows = spark.sql(f"""
            SELECT entity, status, rows_inserted, rows_updated
            FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
            WHERE layer = 'Raw-to-Bronze' AND parameters LIKE '%{batch}%'
              AND start_time >= TIMESTAMP '{PIPELINE_START.strftime("%Y-%m-%d %H:%M:%S")}'
        """).collect()
        res = {r["entity"]: {"inserted": r["rows_inserted"] or 0, "updated": r["rows_updated"] or 0,
                             "status": r["status"]} for r in rows}
        good = len(res) == 3 and all(v["status"] == "SUCCESS" for v in res.values())
        out = {"batch_id": batch, "all_ok": good, "results": res, "from_log_table": True}
        print("  (no result message from 01, used the log table instead)")
    if ok and isinstance(out, dict) and out.get("all_ok"):
        bronze_results.append((batch, out))
        print("  ok:", {k: (v["inserted"], v["updated"]) for k, v in out["results"].items()})
    else:
        failed_windows.append((i, fmt(a), fmt(b), out if not ok else out.get("results")))
        print("  PROBLEM in this window:", out)

print(f"\nBronze finished. windows ok: {len(bronze_results)}, windows with problems: {len(failed_windows)}")


--- window 1/1: 2026-10-23T15:08:08Z -> (now)  (batch incremental_20261024T150808_w01) ---
  ok: {'commits': (0, 18), 'issues': (0, 202), 'pulls': (0, 146)}

Bronze finished. windows ok: 1, windows with problems: 0


## Step 2: Bronze to Silver
One window: only that batch. Many windows: one pass over all of Bronze.
Either way it is `MERGE INTO`, so it is safe to repeat.

In [0]:
silver_ok, silver_out = True, None
if bronze_results:
    silver_batch = bronze_results[0][0] if len(WINDOWS) == 1 else ""
    silver_ok, silver_out = run_child("./02_bronze_to_silver", {"batch_id": silver_batch})
    if silver_ok and isinstance(silver_out, dict) and silver_out.get("all_ok") is False:
        silver_ok, silver_out = False, silver_out.get("problems")
    print("Silver:", "ok" if silver_ok else f"FAILED: {silver_out}")
else:
    silver_ok = False
    print("Nothing reached Bronze, so Silver was skipped.")

Silver: ok


## Step 3: Silver to Gold
Gold only reads Silver rows that were written or refreshed since this
pipeline started (`silver_since`), so it never redoes old work.

In [0]:
gold_ok, gold_out = True, None
if RUN_GOLD and silver_ok:
    # 5 minutes of safety room, in case the clocks differ by a few seconds.
    # Doing a little extra work is harmless: Gold uses MERGE.
    gold_ok, gold_out = run_child("./04_silver_to_gold", {
        "silver_since": (PIPELINE_START - timedelta(minutes=5)).strftime(ISO),
        "batch_id": PREFIX,
    })
    if gold_ok and isinstance(gold_out, dict) and gold_out.get("all_ok") is False:
        gold_ok, gold_out = False, gold_out.get("problems")
    print("Gold:", "ok" if gold_ok else f"FAILED: {gold_out}")
elif not RUN_GOLD:
    print("Gold skipped (run_gold = no).")
else:
    gold_ok = False
    print("Gold skipped because Silver did not finish.")

Gold: ok


## Write one log row for the whole chain

In [0]:
all_ok = (not failed_windows) and silver_ok and gold_ok
problems = []
if failed_windows:
    problems.append("windows to re-run: " + "; ".join(f"{a} -> {b or 'now'}" for _, a, b, _ in failed_windows))
if not silver_ok:
    problems.append(f"silver: {silver_out}")
if not gold_ok:
    problems.append(f"gold: {gold_out}")

ins = sum(v["inserted"] for _, out in bronze_results for v in out["results"].values())
upd = sum(v["updated"] for _, out in bronze_results for v in out["results"].values())
log_run(PIPE_RUN_ID, "End-to-End", "pipeline",
        {"load_type": LOAD_TYPE, "since": fmt(SINCE), "until": fmt(UNTIL), "chunk_days": CHUNK_DAYS,
         "windows": len(WINDOWS), "run_gold": RUN_GOLD, "batch_prefix": PREFIX},
        PIPELINE_START, now_utc(), "SUCCESS" if all_ok else "FAILED",
        rows_inserted=ins, rows_updated=upd, error_message=" | ".join(problems) or None)

print("\nRESULT:", "everything worked" if all_ok else "NOT everything worked")
for p in problems:
    print("  -", p)

[log] End-to-End / pipeline -> SUCCESS (inserted=0, updated=366, quarantined=0, aggregated=0, seconds=265.6)

RESULT: everything worked


In [0]:
display(spark.sql(f"""
    SELECT run_id, layer, entity, status, rows_inserted, rows_updated, rows_quarantined,
           rows_aggregated, duration_seconds, start_time
    FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
    WHERE start_time >= TIMESTAMP '{PIPELINE_START.strftime("%Y-%m-%d %H:%M:%S")}'
    ORDER BY start_time
"""))

run_id,layer,entity,status,rows_inserted,rows_updated,rows_quarantined,rows_aggregated,duration_seconds,start_time
4cc99322-52f6-4d09-9b83-ceefb3d02ba1,End-to-End,pipeline,SUCCESS,0,366,0,0,265.625225,2026-10-24T15:08:08.657Z
e84cd869-a6ea-4586-a2b8-be98f9fd9412,Raw-to-Bronze,commits,SUCCESS,0,18,0,0,16.761051,2026-10-24T15:08:19.872Z
e84cd869-a6ea-4586-a2b8-be98f9fd9412,Raw-to-Bronze,issues,SUCCESS,0,202,0,0,13.175207,2026-10-24T15:08:39.245Z
e84cd869-a6ea-4586-a2b8-be98f9fd9412,Raw-to-Bronze,pulls,SUCCESS,0,146,0,0,21.020039,2026-10-24T15:08:54.815Z
a92a2ac1-b1c0-4f86-bf78-40ab4a6a6af5,Bronze-to-Silver,commits,SUCCESS,0,18,0,0,10.920941,2026-10-24T15:09:40.342Z
a92a2ac1-b1c0-4f86-bf78-40ab4a6a6af5,Bronze-to-Silver,issues,SUCCESS,0,57,0,0,8.761766,2026-10-24T15:09:53.242Z
a92a2ac1-b1c0-4f86-bf78-40ab4a6a6af5,Bronze-to-Silver,pulls,SUCCESS,0,146,0,0,9.81087,2026-10-24T15:10:03.610Z
a3b43234-8b56-4dc1-a05b-6d913c491343,Silver-to-Gold,dim_repo,SUCCESS,0,0,0,9,5.054162,2026-10-24T15:10:43.528Z
a3b43234-8b56-4dc1-a05b-6d913c491343,Silver-to-Gold,dim_date,SUCCESS,0,0,0,8401,4.297918,2026-10-24T15:10:50.553Z
a3b43234-8b56-4dc1-a05b-6d913c491343,Silver-to-Gold,dim_contributor,SUCCESS,0,0,0,8128,6.71536,2026-10-24T15:10:56.385Z


In [0]:
dbutils.notebook.exit(json.dumps({"all_ok": all_ok, "problems": problems, "windows": len(WINDOWS)}))